# Judge Alignment with MemAlign

This notebook aligns our evaluation judge with SME (Subject Matter Expert) feedback using the MemAlign optimizer.

**Prerequisites:**
- Run `00_setup.ipynb` to generate configuration
- Run `04-Evaluation.ipynb` to produce evaluation traces tagged with `eval: complete`
- Complete labeling sessions in the Review App (SME feedback)

**What this notebook does:**
1. Loads evaluation traces with SME feedback
2. Creates a MemAlignOptimizer to distill guidelines from SME annotations
3. Aligns the base judge to produce an aligned judge that reflects organizational preferences
4. Inspects the distilled semantic and episodic memories
5. Registers the aligned judge for use in prompt optimization

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" dspy databricks-mcp langgraph-checkpoint-postgres "psycopg[binary,pool]" databricks-langchain langgraph
dbutils.library.restartPython()

In [ ]:
import json
from pathlib import Path
import mlflow
from mlflow.genai.datasets import get_dataset

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Extract configuration variables
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
DATASET_NAME = CONFIG["evaluation"]["dataset_name"]
JUDGE_MODEL = CONFIG["llm"]["judge_model"]
REFLECTION_MODEL = CONFIG["prompt_registry"]["reflection_model"]
EMBEDDING_MODEL = CONFIG["judges"]["embedding_model"]
ALIGNED_JUDGE_NAME = CONFIG['judges']['aligned_judge_name']

# Set the MLflow experiment
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

# Get traces from the dataset (traces are tagged with "eval: complete" during evaluation)
traces_for_alignment = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    filter_string="tag.eval = 'complete'",
    return_type="list"
)
print(f'Found {len(traces_for_alignment)} traces from evaluation dataset: {DATASET_NAME}')

In [ ]:
import logging

# Set MLflow's logger to only show Errors, not Warnings
logging.getLogger("mlflow").setLevel(logging.ERROR)

## Create MemAlign Optimizer

The MemAlignOptimizer distills guidelines from human feedback (semantic memory) and stores representative examples (episodic memory) to calibrate the judge.

In [ ]:
import mlflow
from mlflow.genai.judges import make_judge
from mlflow.genai.judges.optimizers import MemAlignOptimizer

# Create the MemAlign optimizer
optimizer = MemAlignOptimizer(
    reflection_lm=REFLECTION_MODEL,  # Model for distilling guidelines
    retrieval_k=3,  # Number of similar examples to retrieve
    embedding_model=EMBEDDING_MODEL,  # Model for episodic memory embeddings (from config)
)

print(f"Created MemAlignOptimizer")
print(f"  reflection_lm: {optimizer._reflection_lm}")
print(f"  retrieval_k: {optimizer._retrieval_k}")
print(f"  embedding_model: {optimizer._embedding_model}")

## Load Base Judge and Run Alignment

In [ ]:
from mlflow.genai.scorers import get_scorer

# Load the base judge registered during evaluation
base_judge = get_scorer(name=ALIGNED_JUDGE_NAME)
print(f"Loaded base judge: {base_judge.name}")

In [ ]:
# Align the judge using traces with SME feedback
aligned_judge = base_judge.align(traces=traces_for_alignment, optimizer=optimizer)
print(f"Alignment complete for judge: {aligned_judge.name}")

## Inspect Alignment Memories

Examine the distilled guidelines (semantic memory) and stored examples (episodic memory) produced by MemAlign.

In [ ]:
# Inspect semantic memory (distilled guidelines)
print("=" * 60)
print("SEMANTIC MEMORY (Distilled Guidelines)")
print("=" * 60)
for i, guideline in enumerate(aligned_judge._semantic_memory, 1):
    print(f"\n{i}. {guideline.guideline_text}")
    if guideline.source_trace_ids:
        print(f"   Source traces: {guideline.source_trace_ids[:3]}..." if len(guideline.source_trace_ids) > 3 else f"   Source traces: {guideline.source_trace_ids}")

In [ ]:
# Inspect episodic memory (stored examples)
print("=" * 60)
print("EPISODIC MEMORY (Stored Examples)")
print("=" * 60)
print(f"Total examples: {len(aligned_judge._episodic_memory)}")
print(aligned_judge._episodic_memory)

In [ ]:
# Show the aligned instructions (includes distilled guidelines)
print("=" * 60)
print("ALIGNED JUDGE INSTRUCTIONS")
print("=" * 60)
print(aligned_judge.instructions)

## Update Aligned Judge

In [ ]:
from mlflow.genai.scorers import ScorerSamplingConfig

mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

# Update the aligned judge in the experiment (MemAlign updates in place, no separate registration needed)
try:
    registered_aligned_judge = aligned_judge.update(
        experiment_id=EXPERIMENT_ID,
        sampling_config=ScorerSamplingConfig(sample_rate=0.0)
    )
    print(f"Updated aligned judge: {ALIGNED_JUDGE_NAME}")
except Exception as e:
    print(f"Warning updating aligned judge: {e}")

## (Optional) Re-run Evaluation with Aligned Judge

You can re-run the evaluation using the aligned judge to compare scores with the base judge.

In [ ]:
# Verify we can load the aligned judge back from the experiment
mem_load_judge = get_scorer(name=ALIGNED_JUDGE_NAME)

# Inspect semantic memory (distilled guidelines)
print("=" * 60)
print("RELOADED SEMANTIC MEMORY (Distilled Guidelines)")
print("=" * 60)
for i, guideline in enumerate(mem_load_judge._semantic_memory, 1):
    print(f"\n{i}. {guideline.guideline_text}")
    if guideline.source_trace_ids:
        print(f"   Source traces: {guideline.source_trace_ids[:3]}..." if len(guideline.source_trace_ids) > 3 else f"   Source traces: {guideline.source_trace_ids}")

## Next Steps

The aligned judge is now ready for use in:
- **06-PromptOptimization.ipynb** - Use as the scorer for GEPA prompt optimization
- **07-AgentSkillsGeneration.ipynb** - Guides skill quality evaluation
- **09-Evaluation.ipynb** - Scores the held-out evaluation of both agents